In [5]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(len(v))   
print(v[0])

384
-0.020582036807885073


In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(len(documents))

72


In [7]:
import numpy as np

target_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc_vector = embedder.encode(target_doc["content"])

similarity = np.dot(v, doc_vector)
print(similarity)

0.361070280302606


In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [9]:
chunk_texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_texts)
print(X.shape)

(295, 384)


In [10]:
scores = X.dot(v)

best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print(best_chunk["filename"])
print(scores[best_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
0.648901732433228


In [11]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query4 = "What metric do we use to evaluate a search engine?"
v4 = embedder.encode(query4)

results4 = vindex.search(v4, num_results=5)

for r in results4:
    print(r["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [12]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

query5 = "How do I store vectors in PostgreSQL?"
v5 = embedder.encode(query5)

vector_results = vindex.search(v5, num_results=5)
text_results = text_index.search(query5, num_results=5)

vector_filenames = [r["filename"] for r in vector_results]
text_filenames = [r["filename"] for r in text_results]

print("Vector results:")
for f in vector_filenames:
    print(" ", f)

print("Text results:")
for f in text_filenames:
    print(" ", f)

only_in_vector = set(vector_filenames) - set(text_filenames)
print("In vector but not text:", only_in_vector)


Vector results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
Text results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
In vector but not text: {'02-vector-search/lessons/08-pgvector.md'}


In [13]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query6 = "How do I give the model access to tools?"
v6 = embedder.encode(query6)

vector_results6 = vindex.search(v6, num_results=5)
text_results6 = text_index.search(query6, num_results=5)

results6 = rrf([vector_results6, text_results6])

for r in results6:
    print(r["filename"])
    

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
